In [ ]:
import pandas as pd 
import numpy as np 
from scipy import signal

# ---------------------------------
# 1) PERIOD ESTIMATION USING FFT
# ---------------------------------

def estimate_period_fft(y, min_period, max_period):
    """
    Estimate period using FFT spectral analysis.
    Finds the dominant frequency in the valid range [min_period, max_period].
    """
    y = np.asarray(y, dtype=float)
    n = len(y)
    
    # Set sensible max_period if not provided
    if max_period is None:
        max_period = max(7, min(n // 4, 1000))
    
    # Data too short for analysis
    if n < min_period * 2:
        return max(min_period, min(n, 8))
    
    # Handle missing values by interpolation
    y_series = pd.Series(y)
    y_filled = y_series.interpolate(limit_direction="both").fillna(method='bfill').fillna(method='ffill').to_numpy()
    
    # Handle constant series
    if np.std(y_filled) < 1e-10:
        return min_period
    
    # Detrend and normalize
    y_detrended = signal.detrend(y_filled)
    y_normalized = (y_detrended - np.mean(y_detrended)) / (np.std(y_detrended) + 1e-10)
    
    # Compute FFT
    fft_vals = np.fft.rfft(y_normalized)
    power_spectrum = np.abs(fft_vals) ** 2
    frequencies = np.fft.rfftfreq(n)
    
    # Convert frequencies to periods
    # Avoid division by zero for DC component
    with np.errstate(divide='ignore', invalid='ignore'):
        periods = 1.0 / frequencies
        periods[0] = np.inf  # DC component has infinite period
    
    # Filter to valid period range
    valid_mask = (periods >= min_period) & (periods <= max_period)
    
    if not np.any(valid_mask):
        return min_period
    
    # Find period with maximum power
    valid_power = power_spectrum[valid_mask]
    valid_periods = periods[valid_mask]
    
    best_period = valid_periods[np.argmax(valid_power)]
    
    return int(np.round(best_period))

# ---------------------------------
# 2) FOLDING (PERIODIC TEMPORAL MATRIX)
# ---------------------------------

def fold_series_to_matrix(y, period):
    """
    Fold 1D series into a (n_blocks, period) matrix. 
    Pads the last block with NaN if needed.
    Returns (M, original_len).
    """
    y = np.asarray(y, dtype=float)
    n = len(y)
    n_blocks = int(np.ceil(n / period))
    pad_len = n_blocks * period - n
    
    if pad_len > 0:
        y = np.concatenate([y, np.full(pad_len, np.nan)])
    
    M = y.reshape(n_blocks, period)
    return M, n

def unfold_matrix_to_series(M, original_len):
    """Inverse of fold: row-wise flatten and trim to original length."""
    y = M.reshape(-1)
    return y[:original_len]

# ----------------------------
# 3) SVD + RANK SELECTION
# ----------------------------

def svd_rank(M_filled, energy=0.9):
    """
    Compute SVD and choose rank r by cumulative explained energy.
    Energy is defined as cumulative sum of singular values.
    """
    U, s, Vt = np.linalg.svd(M_filled, full_matrices=False)
    
    total_energy = np.sum(s)
    if total_energy < 1e-12:
        return U, s, Vt, 1
    
    cum_energy = np.cumsum(s) / total_energy
    r = int(np.searchsorted(cum_energy, energy) + 1)
    r = max(1, min(r, min(M_filled.shape)))
    
    return U, s, Vt, r

# -------------------------------------------
# 4) KNN IMPUTE USING ROW EMBEDDINGS FROM SVD
# -------------------------------------------

def _initial_fill(M):
    """
    Column-wise median fill for initial SVD computation.
    Uses global median as fallback for all-NaN columns.
    """
    M_filled = M.copy()
    
    # Compute column medians
    col_medians = np.nanmedian(M_filled, axis=0)
    
    # Fallback to global median for all-NaN columns
    global_median = np.nanmedian(M_filled)
    if np.isnan(global_median):
        global_median = 0.0
    
    col_medians = np.where(np.isnan(col_medians), global_median, col_medians)
    
    # Fill missing values
    nan_indices = np.where(np.isnan(M_filled))
    M_filled[nan_indices] = np.take(col_medians, nan_indices[1])
    
    return M_filled

def knn_in_latent(M, k=5, energy=0.9, allow_future=True):
    """
    Impute NaNs in M by KNN in SVD latent space.
    
    For each missing cell (i,j):
    - Find k nearest rows to row i in latent space
    - Use only rows where column j is observed
    - Optionally restrict to past rows only (allow_future=False)
    - Weighted average by inverse distance
    """
    # Initial fill for SVD
    M_filled = _initial_fill(M)
    
    # Compute SVD and get row embeddings
    U, s, Vt, r = svd_rank(M_filled, energy=energy)
    Z = U[:, :r] * s[:r]  # Row embeddings in latent space
    
    M_imputed = M.copy()
    T, P = M.shape
    observed_mask = ~np.isnan(M)
    
    for i in range(T):
        # Find missing columns in this row
        missing_cols = np.where(~observed_mask[i])[0]
        if len(missing_cols) == 0:
            continue
        
        zi = Z[i]
        
        # Define candidate rows for neighbors
        if allow_future:
            candidates = np.arange(T)
        else:
            candidates = np.arange(0, i)
            # If no past data available, use all rows
            if len(candidates) == 0:
                candidates = np.arange(T)
        
        for j in missing_cols:
            # Filter candidates to those with observed value in column j
            valid_candidates = candidates[observed_mask[candidates, j]]
            
            if len(valid_candidates) == 0:
                # No observed values available, keep initial fill
                continue
            
            # Compute distances in latent space
            distances = np.linalg.norm(Z[valid_candidates] - zi[None, :], axis=1)
            distances = distances + 1e-8  # Avoid division by zero
            
            # Select k nearest neighbors
            if len(distances) > k:
                nearest_indices = np.argpartition(distances, k)[:k]
                neighbor_rows = valid_candidates[nearest_indices]
                neighbor_distances = distances[nearest_indices]
            else:
                neighbor_rows = valid_candidates
                neighbor_distances = distances
            
            # Compute weighted average
            weights = 1.0 / neighbor_distances
            values = M[neighbor_rows, j]
            
            # Additional safety check (shouldn't be needed)
            valid_mask = ~np.isnan(values)
            if not np.any(valid_mask):
                continue
            
            values = values[valid_mask]
            weights = weights[valid_mask]
            
            M_imputed[i, j] = np.sum(weights * values) / np.sum(weights)
    
    return M_imputed

# ----------------------------
# 5) MAIN PIPELINE
# ----------------------------

def impute_throughput_svd_knn(df, col="throughput_bps", min_period=4, max_period=None, 
                               energy=0.9, k=5, allow_future=True):
    """
    Full SVD-KNN imputation pipeline with FFT-based period detection:
    
    1) Detect period using FFT spectral analysis
    2) Fold series into (n_blocks, period) matrix
    3) Compute SVD to get latent row embeddings
    4) Impute missing values using KNN in latent space
    5) Unfold back to original 1D series
    
    Parameters:
    -----------
    df : pd.DataFrame
        Input dataframe
    col : str
        Column name to impute
    min_period : int
        Minimum period to consider
    max_period : int or None
        Maximum period to consider
    energy : float
        Cumulative energy threshold for SVD rank selection (0-1)
    k : int
        Number of nearest neighbors for imputation
    allow_future : bool
        Whether to use future observations as neighbors
    
    Returns:
    --------
    df_imputed : pd.DataFrame
        DataFrame with imputed values
    """
    if col not in df.columns:
        raise ValueError(f"Column '{col}' not found in DataFrame.")
    
    y = df[col].to_numpy(dtype=float)
    
    # Step 1: Detect period using FFT
    try:
        period = estimate_period_fft(y, min_period=min_period, max_period=max_period)
    except Exception as e:
        raise RuntimeError(f"Error in FFT period detection: {str(e)}")
    
    # Step 2: Fold into matrix
    try:
        M, orig_len = fold_series_to_matrix(y, period=period)
    except Exception as e:
        raise RuntimeError(f"Error in folding: {str(e)}")
    
    # Step 3-4: Impute using KNN in latent space
    try:
        M_imputed = knn_in_latent(M, k=k, energy=energy, allow_future=allow_future)
    except Exception as e:
        raise RuntimeError(f"Error in KNN imputation: {str(e)}")
    
    # Step 5: Unfold back to series
    try:
        y_imputed = unfold_matrix_to_series(M_imputed, original_len=orig_len)
    except Exception as e:
        raise RuntimeError(f"Error in unfolding: {str(e)}")
    
    # Create output dataframe
    df_imputed = df.copy()
    df_imputed[col] = y_imputed
    
    return df_imputed

import pandas as pd

# --- 1) Carregar CSV de exemplo ---
df = pd.read_csv("intervalos vazao  esmond data psmp-gn-bw-poz-pl.geant.org to pspmp-anella.csuc.cat 10-24-2023.csv", sep=",")

# Remover espaços extras nas colunas string
df["Data"] = df["Data"].astype(str).str.strip()
df["Intervalo"] = df["Intervalo"].astype(str).str.strip()

# Converter datas (dia vem primeiro)
df["Data"] = pd.to_datetime(df["Data"], dayfirst=True, errors="coerce")

# Criar timestamp usando início do intervalo
df["Hora_inicio"] = df["Intervalo"].str.split(" a ").str[0].str.strip()
df["timestamp"] = pd.to_datetime(df["Data"].dt.strftime("%Y-%m-%d") + " " + df["Hora_inicio"],
                                 errors="coerce")

# Ordenar e definir índice temporal
df = df.sort_values("timestamp").set_index("timestamp")

# Substituir -1 por NaN
df["Vazao"] = df["Vazao"].replace(-1, pd.NA)

# Limpeza antes do pipeline
df["Vazao"] = (
    df["Vazao"]
    .astype(str).str.strip().str.replace(",", ".", regex=False)  # se houver vírgula decimal
    .replace({"": None, "-1": None})                              # trata strings vazias e -1
)
df["Vazao"] = pd.to_numeric(df["Vazao"], errors="coerce")         # vira float com NaN

# (opcional) checar linhas que não viraram número
mask_bad = df["Vazao"].isna()
if mask_bad.any():
    print("Linhas com Vazao inválida (viraram NaN):")
    print(df.loc[mask_bad].head())

# Agora chame o pipeline normalmente
df_imp = impute_throughput_svd_knn(
    df, col="Vazao", min_period=4, max_period=200, energy=0.9, k=5, allow_future=True
)


# --- 2) Aplicar pipeline SVD+KNN ---
df_imp = impute_throughput_svd_knn(
    df,
    col="Vazao",
    min_period=4,
    max_period=200,
    energy=0.9,
    k=5,
    allow_future=True
)



# --- 3) Salvar resultado ---
df_imp.to_csv("resultado2.csv", index=True)
print("\nArquivo 'resultado2.csv' salvo com sucesso!")


Linhas com Vazao inválida (viraram NaN):
                          Data            Intervalo  Vazao Hora_inicio
timestamp                                                             
2023-04-27 00:00:00 2023-04-27  00:00:00 a 05:59:59    NaN    00:00:00
2023-04-27 06:00:00 2023-04-27  06:00:00 a 11:59:59    NaN    06:00:00
2023-04-27 12:00:00 2023-04-27  12:00:00 a 17:59:59    NaN    12:00:00
2023-06-01 00:00:00 2023-06-01  00:00:00 a 05:59:59    NaN    00:00:00
2023-06-01 06:00:00 2023-06-01  06:00:00 a 11:59:59    NaN    06:00:00

Arquivo 'resultado3.csv' salvo com sucesso!


/tmp/ipykernel_16230/1869159388.py:27: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  y_filled = y_series.interpolate(limit_direction="both").fillna(method='bfill').fillna(method='ffill').to_numpy()
